In [1]:
print("Kernel works")

Kernel works


In [2]:
import pandas as pd
from rdkit import Chem

df = pd.read_csv ('/home/susan/mof-co2-adsorption/data/processed/df_chem.csv')
print(df.columns)
print("Dataset shape:", df.shape)
print("Missing MOFID:", df["mofid"].isna().sum())
print("Unique MOFID:", df["mofid"].nunique())


Index(['filename', 'lcd', 'pld', 'void_fraction', 'surface_area_m2g', 'mofid',
       'CO2_uptake_0.01bar_molkg', 'CO2_uptake_0.05bar_molkg',
       'CO2_uptake_0.1bar_molkg', 'CO2_uptake_0.5bar_molkg',
       'CO2_uptake_2.5bar_molkg'],
      dtype='object')
Dataset shape: (27706, 11)
Missing MOFID: 0
Unique MOFID: 24958


In [3]:
print("Exact duplicate rows:", df.duplicated().sum())

Exact duplicate rows: 0


---
we want to answer three questions:

- How many extra occurrences are caused by repeated MOFIDs?
- How many different MOFID strings appear more than once?
- How many times can one MOFID occur?

Step 1 — Count occurrence of each MOFID.


In [4]:
mofid_counts = df["mofid"].value_counts()
# Keep only MOFIDs that appear more than once
repeated_mofids = mofid_counts[mofid_counts > 1]

print("Total rows:", len(df))
print("Unique MOFIDs:", df["mofid"].nunique())
print("Repeated occurrences:", df["mofid"].duplicated().sum())

print(mofid_counts.head())


Total rows: 27706
Unique MOFIDs: 24958
Repeated occurrences: 2748
mofid
* MOFid-v1.NA.NA                                                                         1259
* MOFid-v1.NA.NAno_mof                                                                    495
N#N.[O-]C(=O)C#CC(=O)[O-].[Zn][Zn] MOFid-v1.ERROR.cat0                                     39
N#N.[Cu][Cu].[O-]C(=O)C#CC(=O)[O-] MOFid-v1.ERROR.cat0                                     32
[O-]C(=O)C#CC(=O)[O-].[O-]C(=O)C=CC(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat1      21
Name: count, dtype: int64


| Output                                | Meaning                                                                                                                                             |
| ------------------------------------- | --------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Total rows: 27,706**                |  dataset contains 27,706 MOF records with a non-null value in the `mofid` column.                                                               |
| **Unique MOFIDs: 24,958**             | There are 24,958 different `mofid` strings among those 27,706 rows.                                                                                 |
| **Extra repeated occurrences: 2,748** | After keeping the first occurrence of every MOFID, there are 2,748 additional occurrences of already-seen MOFID strings. This is `27,706 − 24,958`. |
| **Unique MOFIDs that repeat: 616**    | There are 616 different MOFID strings that occur at least twice.                                                                                    |
| **Maximum occurrence: 1,259**         | The most frequently occurring MOFID string appears in 1,259 rows.                                                                                   |


Total rows: 27706
Unique MOFIDs: 24958
Extra repeated occurrences: 2748
Unique MOFIDs that repeat: 616 `number of repeated MOFID types`
Maximum occurrence of one MOFID: 1259 `highest frequency of one MOFID type`
mofid

---


`* MOFid-v1.NA.NA    1259`
means this exact string occurs in 1,259 rows. NA indicates that a normal MOFID representation was not available/generated, so these are not 1,259 copies of the same chemical structure.

---
*MOFid-v1.NA.NAno_mof   495*
means this exact status-like string occurs 495 times. Again, this is not a normal chemical MOFID.

---

*N#N.[O-]C(=O)C#CC(=O)[O-].[Zn][Zn] MOFid-v1.ERRORcat0  39*

means the exact same MOFID string occurs 39 times. Unlike the NA entries, it contains chemical fragments (N#N, linker, Zn) but its MOFID metadata contains ERROR.

---

*N#N.[Cu][Cu].[O-]C(=O)C#CC(=O)[O-] MOFid-v1.ERRORcat0   32*
is another specific chemical representation occurring 32 times, also carrying an ERROR status.

---
*[O-]C(=O)C#CC(=O)[O-].[O-]C(=O)C=CC(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat1      21*
occurs 21 times. This looks different because pcu is a topology designation rather than an NA or ERROR marker.

*The key distinction is:*
27,706 rows ≠ 27,706 unique MOFIDs. There are 24,958 unique MOFID strings, and the repetition is strongly influenced by placeholder/error values such as the 1,259 NA records.

removing both
* MOFid-v1.NA.NA
          ^^^^^^^^^^

* MOFid-v1.NA.NAno_mof
          ^^^^^^^^^^

In [5]:
# removing both
# MOFid-v1.NA.NA
#  MOFid-v1.NA.NAno_mof
df = df[
    ~df["mofid"].str.contains("MOFid-v1.NA", na=False)
].copy()

print("Remaining rows:", len(df))
#Then verify:
print(df.shape)
print(df["mofid"].str.contains("MOFid-v1.NA", na=False).sum())


Remaining rows: 25952
(25952, 11)
0


In [6]:
# quantify only the ERROR records:
error_count = df["mofid"].str.contains(
    "MOFid-v1.ERROR", na=False
).sum()

print("ERROR-type MOFIDs:", error_count)

# take Take one ERROR MOFID to check it works with rdkit
error_example = df[
    df["mofid"].str.contains("MOFid-v1.ERROR", na=False)
]["mofid"].iloc[0]

print(error_example)

# extract it :
chemical_smiles = error_example.split(" ")[0]
print(chemical_smiles)

# Test with RDKIT 
mol = Chem.MolFromSmiles(chemical_smiles)
print(mol)

ERROR-type MOFIDs: 1691
N#Cc1ccc2c(c1)C1([CH]C(=C2C(=O)[O-])C#N)C(=O)ON1[C]C1=C[C]2[C]N=N[C]c3c(C#N)cc4c(c3C#N)c3c(C#N)c(C(=O)[O-])c(-c5cccc([C]N=N[C]c6c(c(-c7cc8C(=O)OC1C(=C2)c8cc7C(=O)[O-])ccc6)C#N)c5)cc3c(=O)oc4=[N].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.ERROR
N#Cc1ccc2c(c1)C1([CH]C(=C2C(=O)[O-])C#N)C(=O)ON1[C]C1=C[C]2[C]N=N[C]c3c(C#N)cc4c(c3C#N)c3c(C#N)c(C(=O)[O-])c(-c5cccc([C]N=N[C]c6c(c(-c7cc8C(=O)OC1C(=C2)c8cc7C(=O)[O-])ccc6)C#N)c5)cc3c(=O)oc4=[N].[Zn][O]([Zn])([Zn])[Zn]
None


[08:58:24] Explicit valence for atom # 89 O, 3, is greater than permitted


In [7]:
# Now test all 1,691 ERROR records with RDkit : 
error_mofids = df[
    df["mofid"].str.contains("MOFid-v1.ERROR", na=False)
]["mofid"]

parsed = 0
failed = 0

for mofid in error_mofids:
    chemical_smiles = mofid.split(" ")[0]
    mol = Chem.MolFromSmiles(chemical_smiles)

    if mol is None:
        failed += 1
    else:
        parsed += 1

print("Total ERROR records:", len(error_mofids))
print("Successfully parsed:", parsed)
print("Failed to parse:", failed)

[08:58:24] Explicit valence for atom # 89 O, 3, is greater than permitted
[08:58:24] Explicit valence for atom # 25 O, 3, is greater than permitted
[08:58:24] Explicit valence for atom # 33 O, 3, is greater than permitted
[08:58:24] Explicit valence for atom # 23 O, 3, is greater than permitted
[08:58:24] Explicit valence for atom # 13 O, 3, is greater than permitted
[08:58:24] Explicit valence for atom # 31 O, 3, is greater than permitted
[08:58:24] Explicit valence for atom # 101 O, 3, is greater than permitted
[08:58:24] Explicit valence for atom # 33 O, 3, is greater than permitted
[08:58:24] Explicit valence for atom # 63 O, 3, is greater than permitted
[08:58:24] Explicit valence for atom # 51 O, 3, is greater than permitted
[08:58:24] Explicit valence for atom # 55 O, 3, is greater than permitted
[08:58:24] Explicit valence for atom # 54 O, 3, is greater than permitted
[08:58:24] Explicit valence for atom # 51 O, 3, is greater than permitted
[08:58:24] Explicit valence for atom 

Total ERROR records: 1691
Successfully parsed: 1628
Failed to parse: 63


[08:58:24] Explicit valence for atom # 133 O, 3, is greater than permitted
[08:58:24] Explicit valence for atom # 64 N, 4, is greater than permitted
[08:58:24] Explicit valence for atom # 15 N, 4, is greater than permitted
[08:58:24] Explicit valence for atom # 197 O, 3, is greater than permitted
[08:58:24] Explicit valence for atom # 19 N, 4, is greater than permitted
[08:58:24] Explicit valence for atom # 4 N, 4, is greater than permitted
[08:58:24] Explicit valence for atom # 3 N, 4, is greater than permitted
[08:58:24] Explicit valence for atom # 137 O, 3, is greater than permitted
[08:58:24] Explicit valence for atom # 89 O, 3, is greater than permitted
[08:58:24] Explicit valence for atom # 12 O, 3, is greater than permitted
[08:58:24] Explicit valence for atom # 37 O, 3, is greater than permitted
[08:58:24] Explicit valence for atom # 37 O, 3, is greater than permitted
[08:58:24] Explicit valence for atom # 78 O, 3, is greater than permitted
[08:58:24] Explicit valence for atom 

---

*27,706 non-null MOFID rows → remove 1,754 NA placeholders → 25,952 rows → 1,691 ERROR-labelled records → 1,628 parse successfully and only 63 fail.*

---




### Can RDKit parse the chemical representation for all 25,952 remaining records?

In [8]:
print("Dataset shape:", df.shape)

Dataset shape: (25952, 11)


In [9]:
# first look at one MOFID from  dataset:
mofid = df["mofid"].iloc[0]

print(mofid)

[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat0


---
I has two part chemical representation and MOFid metatdata:  `MOFid-v1.pcu.cat0`

---

In [10]:
# .split(" ") separates the MOFID string at the space.
# [0] selects the chemical representation before the MOFID metadata.
chemical_smiles = mofid.split(" ")[0]

print(chemical_smiles)

from rdkit import Chem

mol = Chem.MolFromSmiles(chemical_smiles)

print(mol)

[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn]
None


[08:58:24] Explicit valence for atom # 13 O, 3, is greater than permitted


----
whether we can isolate the linker from your MOFID and process that linker with RDKit.

----

In [11]:
fragments = chemical_smiles.split(".")

print(fragments)
fragment = Chem.MolFromSmiles(fragments[0])

print(fragment)

['[O-]C(=O)c1ccc(cc1)C(=O)[O-]', '[Zn][O]([Zn])([Zn])[Zn]']


---
<rdkit.Chem.rdchem.Mol object at 0x75c18c10e8f0>

RDKit successfully parsed the first fragment.

---

In [12]:
# Check whole dataset with RDKit 
failed_mofids = []
parsed = 0 # number RDKit successfully reads
failed = 0 # number RDKit cannot read
for mofid in df["mofid"]:
    chemical_smiles = mofid.split(" ")[0]
    mol = Chem.MolFromSmiles(chemical_smiles)

    if mol is None:
        failed += 1
        failed_mofids.append(mofid)   
    else:
        parsed += 1

print("Successfully parsed:", parsed)
print("Failed to parse:", failed)

[08:58:24] Explicit valence for atom # 13 O, 3, is greater than permitted
[08:58:24] Explicit valence for atom # 13 O, 3, is greater than permitted
[08:58:24] Explicit valence for atom # 16 O, 3, is greater than permitted
[08:58:24] Explicit valence for atom # 31 O, 3, is greater than permitted
[08:58:24] Explicit valence for atom # 27 O, 3, is greater than permitted
[08:58:24] Explicit valence for atom # 30 O, 3, is greater than permitted
[08:58:24] Explicit valence for atom # 24 O, 3, is greater than permitted
[08:58:24] Explicit valence for atom # 57 O, 3, is greater than permitted
[08:58:24] Explicit valence for atom # 57 O, 3, is greater than permitted
[08:58:24] Explicit valence for atom # 27 O, 3, is greater than permitted
[08:58:24] Explicit valence for atom # 65 O, 3, is greater than permitted
[08:58:24] Explicit valence for atom # 69 O, 3, is greater than permitted
[08:58:24] Explicit valence for atom # 94 O, 3, is greater than permitted
[08:58:24] Explicit valence for atom #

Successfully parsed: 12940
Failed to parse: 13012


[08:58:30] Explicit valence for atom # 46 O, 3, is greater than permitted
[08:58:30] Explicit valence for atom # 46 O, 3, is greater than permitted
[08:58:30] Explicit valence for atom # 46 O, 3, is greater than permitted
[08:58:30] Explicit valence for atom # 45 O, 3, is greater than permitted
[08:58:30] Explicit valence for atom # 48 O, 3, is greater than permitted
[08:58:30] Explicit valence for atom # 46 O, 3, is greater than permitted
[08:58:30] Explicit valence for atom # 50 O, 3, is greater than permitted
[08:58:30] Explicit valence for atom # 48 O, 3, is greater than permitted
[08:58:30] Explicit valence for atom # 45 O, 3, is greater than permitted
[08:58:30] Explicit valence for atom # 44 O, 3, is greater than permitted
[08:58:30] Explicit valence for atom # 43 O, 3, is greater than permitted
[08:58:30] Explicit valence for atom # 41 O, 3, is greater than permitted
[08:58:30] Explicit valence for atom # 41 O, 3, is greater than permitted
[08:58:30] Explicit valence for atom #

---
Can standard RDKit parse all 25,952 complete chemical representations?

No. It parses 12,940, while 13,012 fail.

Next Step: should be only to collect the failed MOFIDs so we can inspect what causes these failures.

----

In [13]:
# collect failed mofid to understand about their features:
# We check a small batch of dataset
for mofid in failed_mofids[:10]:
    print(mofid)
    print()

[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat0

[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat1

[O-]C(=O)c1cc(F)c(c(c1F)F)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat1

COc1cc(cc(c1C(=O)[O-])OC)C(=O)[O-].COc1cc(ccc1C(=O)[O-])C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat0

CCc1cc(C(=O)[O-])c(c(c1C(=O)[O-])CC)CC.[O-]C(=O)C#CC(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat1

[O-]C(=O)c1cc(Br)c2c(c1)ccc(c2)C(=O)[O-].[O-]C(=O)C(=O)[O-].[O-]C(=O)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.UNKNOWN.cat0

[O-]C(=O)C(=O)[O-].[O-]C(=O)c1ccc2c(c1)ccc(c2C)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat0

[O-]C(=O)c1cc2ccc(c(c2c(c1C)C)C)C(=O)[O-].[O-]C(=O)c1ccc2c(c1)cc(c(c2)C(=O)[O-])C.[O-]C(=O)c1ccc2c(c1C)c(C)c(c(c2C)C(=O)[O-])C.[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat0

[O-]C(=O)c1cc2ccc(c(c2c(c1C)C)C)C(=O)[O-].[O-]C(=O)c1ccc2c(c1)cc(c(c2)C(=O)[O-])C.[O-]C(=O)c1ccc2c(c1C)c(C)c(c(c2C)C(=O)[O-])C.[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat1

-----

This supports hypothesis that RDKit may be rejecting the Zn–O coordination fragment, while the organic linker itself may still be readable.

-----

In [14]:
# How many of the 13,012 failed MOFIDs contain this exact Zn–O fragment?
# How many of the 13,012 failed MOFIDs contain this exact Zn–O fragment?

zn_o_cluster = "[Zn][O]([Zn])([Zn])[Zn]"

with_zn_o_cluster = 0
without_zn_o_cluster = 0

for mofid in failed_mofids:
    if zn_o_cluster in mofid:
        with_zn_o_cluster += 1
    else:
        without_zn_o_cluster += 1

print(
    f"With Zn-O cluster: {with_zn_o_cluster}, "
    f"Without Zn-O cluster: {without_zn_o_cluster}"
)

With Zn-O cluster: 12284, Without Zn-O cluster: 728


In [15]:
without_zn_o_cluster = []
for mofid in failed_mofids:
    if zn_o_cluster not in mofid:
        without_zn_o_cluster.append(mofid)

for mofid in without_zn_o_cluster[:10]:
    print(mofid)
    print() 

OC1=[N]=C(C(=N[CH]1)O)O.[O-]C(=O)c1cc(O)c(c(c1)O)C(=O)[O-].[O-]C(=O)c1ccc(c(c1)O)C(=O)[O-].[Zn][Zn] MOFid-v1.pcu.cat0

OC1=[N]=C(C(=N[CH]1)O)O.[O-]C(=O)c1cc(O)c(cc1O)C(=O)[O-].[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][Zn] MOFid-v1.pcu.cat0

OC1=[N]=C(C(=N[CH]1)O)O.[O-]C(=O)c1ccc(c(c1)O)C(=O)[O-].[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][Zn] MOFid-v1.pcu.cat0

OC1=[N]=C(C(=N[CH]1)O)O.[O-]C(=O)c1ccc(c(c1)O)C(=O)[O-].[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][Zn] MOFid-v1.pcu.cat1

CCOC1=[N]=C(C=N[CH]1)OCC.CCOc1cc(cc(c1C(=O)[O-])OCC)C(=O)[O-].[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][Zn] MOFid-v1.pcu.cat0

CCOC1=[N]=C(C(=N[CH]1)OCC)OCC.CCOc1cc(C(=O)[O-])c(cc1C(=O)[O-])OCC.[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][Zn] MOFid-v1.pcu.cat0

CCOC1=[N]=C(C(=N[CH]1)OCC)OCC.CCOc1cc(C(=O)[O-])c(cc1C(=O)[O-])OCC.[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][Zn] MOFid-v1.pcu.cat0

[O-]C(=O)c1ccc(cc1)C(=O)[O-].CCCOC1=[N]=C(C(=N[CH]1)OCCC)OCCC.[O-]C(=O)c1ccc(cc1OCCC)C(=O)[O-].[Zn][Zn] MOFid-v1.pcu.cat0

COC1=[N]=C(OC)C=C([CH]1)c1ccncc1OC.COc1cc(C(=O)[O-]

In [16]:
# get overall overview of data :
error_count = 0
pcu_cat0_count = 0
pcu_cat1_count = 0
n2_count = 0

for mofid in failed_mofids:
    if "ERROR" in mofid:
        error_count += 1
    if "pcu.cat0" in mofid:
        pcu_cat0_count += 1
    if "pcu.cat1" in mofid:
        pcu_cat1_count += 1
    if "N#N" in mofid:
        n2_count += 1

print("ERROR:", error_count)
print("pcu.cat0:", pcu_cat0_count)
print("pcu.cat1:", pcu_cat1_count)
print("Contains N#N:", n2_count)

ERROR: 63
pcu.cat0: 7210
pcu.cat1: 3659
Contains N#N: 6


| Pattern        | Failed MOFIDs |
| -------------- | ------------: |
| `ERROR`        |            63 |
| `pcu.cat0`     |         7,210 |
| `pcu.cat1`     |         3,659 |
| Contains `N#N` |             6 |


In [17]:
# fragment analysis and Linker extraction:
# Step 1 :Extract the chemical portion of each MOFID by removing the MOFid-v1... metadata.
chemical_representations = []

for mofid in df["mofid"]:
    chemical_smiles = mofid.split("MOFid-v1")[0].strip() # space and mofide_v1“For every MOFID, remove the MOFID metadata and store only its chemical representation.”
    chemical_representations.append(chemical_smiles)

for chemical in chemical_representations[:5]:
    print(chemical)
    print()


[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn]

[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn]

[O-]C(=O)c1cc(F)c(c(c1F)F)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn]

COc1cc(cc(c1C(=O)[O-])OC)C(=O)[O-].COc1cc(ccc1C(=O)[O-])C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn]

CCc1cc(C(=O)[O-])c(c(c1C(=O)[O-])CC)CC.[O-]C(=O)C#CC(=O)[O-].[Zn][O]([Zn])([Zn])[Zn]



In [18]:
df["chemical_representation"] = chemical_representations
print(df.columns)
df_fragments = df.copy()

Index(['filename', 'lcd', 'pld', 'void_fraction', 'surface_area_m2g', 'mofid',
       'CO2_uptake_0.01bar_molkg', 'CO2_uptake_0.05bar_molkg',
       'CO2_uptake_0.1bar_molkg', 'CO2_uptake_0.5bar_molkg',
       'CO2_uptake_2.5bar_molkg', 'chemical_representation'],
      dtype='object')


In [19]:
# .	Split the chemical representation at .
#  to obtain individual fragments. 
all_fragments = []

for chemicals in df_fragments["chemical_representation"]:
    chemical_smiles = chemicals.split(".") #remove [0] to keep all fragments space and mofide_v1“For every MOFID, remove the MOFID metadata and store only its chemical representation.”
    all_fragments.append(chemical_smiles)

for chemical in all_fragments[:5]:
    print(chemical)
    print()

['[O-]C(=O)c1ccc(cc1)C(=O)[O-]', '[Zn][O]([Zn])([Zn])[Zn]']

['[O-]C(=O)c1ccc(cc1)C(=O)[O-]', '[Zn][O]([Zn])([Zn])[Zn]']

['[O-]C(=O)c1cc(F)c(c(c1F)F)C(=O)[O-]', '[Zn][O]([Zn])([Zn])[Zn]']

['COc1cc(cc(c1C(=O)[O-])OC)C(=O)[O-]', 'COc1cc(ccc1C(=O)[O-])C(=O)[O-]', '[Zn][O]([Zn])([Zn])[Zn]']

['CCc1cc(C(=O)[O-])c(c(c1C(=O)[O-])CC)CC', '[O-]C(=O)C#CC(=O)[O-]', '[Zn][O]([Zn])([Zn])[Zn]']



In [20]:
# 3.	Count the number of fragments per MOF to understand how simple or complex the representations are. 

fragment_counts = []
for fragment in all_fragments:
    number_of_fragments = len(fragment)
    fragment_counts.append(number_of_fragments)


for fragment in fragment_counts[:5]:
    print(fragment)
    print()

2

2

2

3

3



In [21]:
#add  all_fragments and fragment_count to dataset
df_fragments["all_fragments"] = all_fragments
df_fragments["fragment_count"] = fragment_counts
print(df_fragments.columns)

Index(['filename', 'lcd', 'pld', 'void_fraction', 'surface_area_m2g', 'mofid',
       'CO2_uptake_0.01bar_molkg', 'CO2_uptake_0.05bar_molkg',
       'CO2_uptake_0.1bar_molkg', 'CO2_uptake_0.5bar_molkg',
       'CO2_uptake_2.5bar_molkg', 'chemical_representation', 'all_fragments',
       'fragment_count'],
      dtype='object')


---

### 4.Identify unique fragments and their frequencies to determine whether a relatively small number of components recur throughout the dataset.

 There is one important issue: all_fragments is currently a list of lists:

MOF 1 → [fragment A, fragment B]
MOF 2 → [fragment A, fragment B]
MOF 3 → [fragment C, fragment B]

---
 

In [22]:
fragment_columns = pd.DataFrame(all_fragments, index=df_fragments.index)
fragment_columns.columns = [
    f"fragment_{i+1}" for i in range(fragment_columns.shape[1])
]

print(fragment_columns.head())

                               fragment_1                      fragment_2  \
0            [O-]C(=O)c1ccc(cc1)C(=O)[O-]         [Zn][O]([Zn])([Zn])[Zn]   
1            [O-]C(=O)c1ccc(cc1)C(=O)[O-]         [Zn][O]([Zn])([Zn])[Zn]   
2     [O-]C(=O)c1cc(F)c(c(c1F)F)C(=O)[O-]         [Zn][O]([Zn])([Zn])[Zn]   
3      COc1cc(cc(c1C(=O)[O-])OC)C(=O)[O-]  COc1cc(ccc1C(=O)[O-])C(=O)[O-]   
4  CCc1cc(C(=O)[O-])c(c(c1C(=O)[O-])CC)CC           [O-]C(=O)C#CC(=O)[O-]   

                fragment_3 fragment_4 fragment_5 fragment_6 fragment_7  \
0                     None       None       None       None       None   
1                     None       None       None       None       None   
2                     None       None       None       None       None   
3  [Zn][O]([Zn])([Zn])[Zn]       None       None       None       None   
4  [Zn][O]([Zn])([Zn])[Zn]       None       None       None       None   

  fragment_8 fragment_9 fragment_10 fragment_11 fragment_12 fragment_13  \
0       None     

In [23]:
fragment_columns["fragment_1"].value_counts()

fragment_1
[O-]C(=O)C#CC(=O)[O-]                                          643
N#N                                                            594
[Cu][Cu]                                                       199
N=N                                                            140
N#Cc1ccc(cc1)C#N                                               107
                                                              ... 
OC1=C(N=Nc2c(O)cnc(c2O)O)[C](C(=N[CH]1)O)O                       1
CCC1=C([C]=c2c(=C1)ccc1c2ccc2c1cc(cc2)C(=O)[O-])C(=O)[O-]        1
Nc1ccc2c(c1)C(=[C]C=C2C(=O)[O-])C(=O)[O-]                        1
CCCC1=C(CCC)C(=C[C]=C1N=Nc1ccc(c(c1)CCC)C(=O)[O-])C(=O)[O-]      1
ClOC(=O)c1c(Cl)cc(c2=CC(=[C][C]=c12)Cl)C(=O)[O-]                 1
Name: count, Length: 8017, dtype: int64

In [24]:
for column in fragment_columns.columns:
    print(column)
    print("Unique fragments:", fragment_columns[column].nunique())
    print(fragment_columns[column].value_counts().head())
    print()

fragment_1
Unique fragments: 8017
fragment_1
[O-]C(=O)C#CC(=O)[O-]    643
N#N                      594
[Cu][Cu]                 199
N=N                      140
N#Cc1ccc(cc1)C#N         107
Name: count, dtype: int64

fragment_2
Unique fragments: 7558
fragment_2
[Cu][Cu]                 2080
[O-]C(=O)C#CC(=O)[O-]     923
[Zn][Zn]                  498
[O-]C(=O)C=CC(=O)[O-]     345
N#N                       304
Name: count, dtype: int64

fragment_3
Unique fragments: 6287
fragment_3
[Zn][O]([Zn])([Zn])[Zn]    2075
[Cu][Cu]                   1728
[Zn][Zn]                    802
[O-]C(=O)C#CC(=O)[O-]       771
[O-]C(=O)C=CC(=O)[O-]       508
Name: count, dtype: int64

fragment_4
Unique fragments: 1772
fragment_4
[Zn][O]([Zn])([Zn])[Zn]    8654
[Zn][Zn]                   6784
[Cu][Cu]                    996
[O]                         353
[O-]C(=O)C#CC(=O)[O-]       246
Name: count, dtype: int64

fragment_5
Unique fragments: 410
fragment_5
[Zn][O]([Zn])([Zn])[Zn]    1002
[Zn][Zn]             

| Position      | Unique fragments | Main observation                                        |
| ------------- | ---------------: | ------------------------------------------------------- |
| fragment_1    |            8,017 | Highly diverse; organic components, `N#N`, metals, etc. |
| fragment_2    |            7,558 | Highly diverse; both organic and metal fragments        |
| fragment_3    |            6,287 | Still diverse; metals becoming more frequent            |
| fragment_4    |            1,772 | Strongly dominated by Zn/Cu fragments                   |
| fragment_5    |              410 | Mostly metal/simple fragments                           |
| fragment_6    |              120 | Much lower diversity                                    |
| fragment_7    |               46 | Mostly recurring metal/simple fragments                 |
| fragment_8    |               20 | Very few types                                          |
| fragment_9–14 |           11 → 1 | Rare, unusually complex MOFs                            |


In [25]:
# should only be identifying metal-containing fragments among the fragments you already created
# df.iloc[0] selects the first row of a Pandas DataFrame using zero-based integer positional indexing
row = fragment_columns.iloc[0]
print(row)


fragment_1     [O-]C(=O)c1ccc(cc1)C(=O)[O-]
fragment_2          [Zn][O]([Zn])([Zn])[Zn]
fragment_3                             None
fragment_4                             None
fragment_5                             None
fragment_6                             None
fragment_7                             None
fragment_8                             None
fragment_9                             None
fragment_10                            None
fragment_11                            None
fragment_12                            None
fragment_13                            None
fragment_14                            None
Name: 0, dtype: object


In [26]:
# To print is not None items for one row
# = means assign
# != means not equal
# for None, the preferred form of is not None.
row = fragment_columns.iloc[0]
for fragment in row:
    if fragment is not None:
        print(fragment)

[O-]C(=O)c1ccc(cc1)C(=O)[O-]
[Zn][O]([Zn])([Zn])[Zn]
